# 06 — Cross-cohort prediction pilot

Nested leave-one-cohort-out evaluation of logistic LASSO and Elastic Net. This is a restricted development pilot, not the final biomarker analysis.

In [1]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount("/content/drive")

candidates = [Path(os.environ["CMPS355_ROOT"])] if os.environ.get("CMPS355_ROOT") else []
candidates.extend([Path("/content/drive/MyDrive/CMPS355_Research"), Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in candidates if (p / ".cmps355_root").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("CMPS355_Research project root not found")
sys.path.insert(0, str(ROOT / "src"))
print("Project:", ROOT.name)

Mounted at /content/drive
Project: CMPS355_Research


## Protocol and data integrity

In [2]:
import json
import numpy as np
import pandas as pd
from cmps355.split_design import read_sample_index
from cmps355.pilot import PilotConfig, load_verified_splits

manifest = json.loads((ROOT / "configs" / "dataset.json").read_text(encoding="utf-8"))
index = read_sample_index(ROOT / manifest["artifact"], manifest)
outer, inner, split_path = load_verified_splits(ROOT, index, manifest)
config = PilotConfig()
print("Dataset samples:", len(index))
print("Verified splits:", split_path.relative_to(ROOT))
print("Outer folds:", outer["outer_heldout"].nunique())
print("Nested folds:", inner[["outer_heldout", "inner_heldout"]].drop_duplicates().shape[0])
print("Pilot configuration:", config)
display(pd.crosstab(index["cohort"], index["class"]))

Dataset samples: 689
Verified splits: runs/split_design/20261107T040042030201Z
Outer folds: 4
Nested folds: 12
Pilot configuration: PilotConfig(c_values=(0.03, 0.3, 3.0), max_genes=300, min_cpm=1.0, min_sample_fraction=0.1, max_iter=1200, tol=0.001, seed=355)


class,CD,Not IBD
cohort,,
d1,218,42
d2,134,45
d3,34,42
d4,109,65


## Nested cross-cohort evaluation

In [3]:
from cmps355.pilot import run_pilot

output_dir, metrics = run_pilot(ROOT, config)
print("Run:", output_dir.relative_to(ROOT))
display(metrics[["test_cohort", "method", "C", "mean_inner_auroc", "auroc", "average_precision", "balanced_accuracy", "sensitivity", "specificity", "n_nonzero_coefficients", "converged"]])

Outer holdout d1: train=429, test=260
  lasso: C=0.03, AUROC=0.832, selected=5, converged=True
  elastic_net: C=0.03, AUROC=0.866, selected=14, converged=True
Outer holdout d2: train=510, test=179
  lasso: C=3, AUROC=0.662, selected=225, converged=True
  elastic_net: C=0.03, AUROC=0.655, selected=28, converged=True
Outer holdout d3: train=613, test=76
  lasso: C=3, AUROC=0.661, selected=251, converged=True
  elastic_net: C=0.3, AUROC=0.734, selected=138, converged=True
Outer holdout d4: train=515, test=174
  lasso: C=0.03, AUROC=0.775, selected=6, converged=True
  elastic_net: C=0.03, AUROC=0.785, selected=20, converged=True
Run: runs/pilot/20261107T041823175243Z


,test_cohort,method,C,mean_inner_auroc,auroc,average_precision,balanced_accuracy,sensitivity,specificity,n_nonzero_coefficients,converged
0,d1,lasso,0.03,0.770463,0.831586,0.965833,0.497706,0.995413,0.000000,5,True
1,d1,elastic_net,0.03,0.742016,0.866099,0.974427,0.495413,0.990826,0.000000,14,True
2,d2,lasso,3.00,0.796818,0.662189,0.845732,0.558624,0.828358,0.288889,225,True
3,d2,elastic_net,0.03,0.817516,0.655390,0.839070,0.627529,0.432836,0.822222,28,True
4,d3,lasso,3.00,0.764955,0.661064,0.617455,0.612745,0.558824,0.666667,251,True
5,d3,elastic_net,0.30,0.769773,0.733894,0.748892,0.668768,0.647059,0.690476,138,True
6,d4,lasso,0.03,0.731102,0.775441,0.867573,0.597812,0.211009,0.984615,6,True
7,d4,elastic_net,0.03,0.729062,0.785180,0.877918,0.616161,0.247706,0.984615,20,True


## Summary and recorded artefacts

In [4]:
report = json.loads((output_dir / "report.json").read_text(encoding="utf-8"))
assert report["status"] == "PASS"
summary = metrics.groupby("method", as_index=False)[["auroc", "average_precision", "balanced_accuracy", "sensitivity", "specificity"]].mean(numeric_only=True)
display(summary)
print("Convergence warnings:", report["n_convergence_warnings"])
print("Saved files:", sorted(p.name for p in output_dir.iterdir()))
print("STAGE 3B PILOT:", report["status"])

,method,auroc,average_precision,balanced_accuracy,sensitivity,specificity
0,elastic_net,0.76014,0.860077,0.601968,0.579607,0.624328
1,lasso,0.73257,0.824148,0.566722,0.648401,0.485043


Convergence warnings: 0
Saved files: ['candidate_genes.csv', 'inner_scores.csv', 'outer_metrics.csv', 'predictions.csv', 'preprocessing.csv', 'report.json', 'selected_genes.csv']
STAGE 3B PILOT: PASS
